In [ ]:
# ── Cell 2: Imports ───────────────────────────────────────────────────────────
import os
import warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

print(f"TensorFlow {tf.__version__}")
print("GPU available:", bool(tf.config.list_physical_devices('GPU')))

In [ ]:
# ── Cell 3: Config ────────────────────────────────────────────────────────────
TRAIN_PATH   = # your folder path for train data 
TEST_PATH    = # your folder path for test data 
CLASS_NAMES  = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
IMG_SIZE     = 48       # keep 48x48 — MobileNetV2 accepts any size ≥ 32
BATCH_SIZE   = 64
EPOCHS       = 100
MODEL_PATH   = 'best_emotion_model_v2.keras'
RESULTS_DIR  = 'results'
os.makedirs(RESULTS_DIR, exist_ok=True)

In [ ]:
# ── Cell 4: Data loading ──────────────────────────────────────────────────────

def load_images_from_folder(folder_path, class_names):
    images, labels = [], []
    for label, emotion in enumerate(class_names):
        emotion_folder = os.path.join(folder_path, emotion)
        if not os.path.isdir(emotion_folder):
            print(f"WARNING: folder not found — {emotion_folder}")
            continue
        for image_name in os.listdir(emotion_folder):
            image_path = os.path.join(emotion_folder, image_name)
            try:
                img = load_img(image_path, target_size=(IMG_SIZE, IMG_SIZE), color_mode='rgb')
                img_array = img_to_array(img) / 255.0
                images.append(img_array)
                labels.append(label)
            except Exception as e:
                print(f"Error loading {image_path}: {e}")
    return np.array(images), np.array(labels)

print("Loading training data...")
X_train, y_train = load_images_from_folder(TRAIN_PATH, CLASS_NAMES)

print("Loading test data...")
X_test, y_test = load_images_from_folder(TEST_PATH, CLASS_NAMES)

print(f"Train: {X_train.shape} | Test: {X_test.shape}")

# Show class distribution so you can see the imbalance
unique, counts = np.unique(y_train, return_counts=True)
print("\nClass distribution (train):")
for cls, cnt in zip(CLASS_NAMES, counts):
    print(f"  {cls:10s}: {cnt}")

In [ ]:
# ── Cell 5: Class weights (NEW) ───────────────────────────────────────────────

class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights = dict(enumerate(class_weights_array))

print("Class weights:")
for cls, w in zip(CLASS_NAMES, class_weights_array):
    print(f"  {cls:10s}: {w:.3f}")

In [ ]:
# ── Cell 6: Model — MobileNetV2 backbone (NEW) ────────────────────────────────

def build_model(num_classes, img_size=48):
    # Pretrained backbone — frozen initially
    backbone = keras.applications.MobileNetV2(
        input_shape=(img_size, img_size, 3),
        include_top=False,          
        weights='imagenet'          
    )
    backbone.trainable = False      

    inputs = keras.Input(shape=(img_size, img_size, 3))
    
    x = keras.applications.mobilenet_v2.preprocess_input(inputs * 255.0)
    x = backbone(x, training=False)
    
    # Classification head
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = keras.Model(inputs, outputs)
    return model, backbone

model, backbone = build_model(len(CLASS_NAMES), IMG_SIZE)
model.compile(
    optimizer=keras.optimizers.Adam(0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

In [ ]:
# ── Cell 7: Data augmentation ─────────────────────────────────────────────────
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rotation_range=15,         
    zoom_range=0.15,
    horizontal_flip=True,
    width_shift_range=0.1,      
    height_shift_range=0.1,    
    brightness_range=[0.8, 1.2] 
)
datagen.fit(X_train)

In [ ]:
# ── Cell 8: Phase 1 Training — head only ─────────────────────────────────────


print("=== Phase 1: Training classification head (backbone frozen) ===")

callbacks = [
    ModelCheckpoint(MODEL_PATH, monitor='val_accuracy', save_best_only=True, mode='max', verbose=1),
    EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-7, verbose=1)
]

history_phase1 = model.fit(
    datagen.flow(X_train, y_train, batch_size=BATCH_SIZE),
    validation_data=(X_test, y_test),
    epochs=50,                 
    class_weight=class_weights, 
    callbacks=callbacks
)

print(f"\nPhase 1 best val_accuracy: {max(history_phase1.history['val_accuracy']):.4f}")

In [ ]:
# ── Cell 9: Phase 2 Training — fine-tune entire model (NEW) ──────────────────

print("=== Phase 2: Fine-tuning entire model (backbone unfrozen) ===")

backbone.trainable = True

model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_ft = [
    ModelCheckpoint(MODEL_PATH, monitor='val_accuracy', save_best_only=True, mode='max', verbose=1),
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-8, verbose=1)
]

history_phase2 = model.fit(
    datagen.flow(X_train, y_train, batch_size=BATCH_SIZE),
    validation_data=(X_test, y_test),
    epochs=EPOCHS,
    class_weight=class_weights,
    callbacks=callbacks_ft
)

print(f"\nPhase 2 best val_accuracy: {max(history_phase2.history['val_accuracy']):.4f}")

In [ ]:
# ── Cell 10: Evaluation ───────────────────────────────────────────────────────

# Load best checkpoint
model = keras.models.load_model(MODEL_PATH)

test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Accuracy : {test_acc:.4f}")
print(f"Test Loss     : {test_loss:.4f}")

y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

In [ ]:
# ── Cell 11: Plots — saved to results ───────────────────────────────────────

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title(f'Confusion Matrix  (Test Acc: {test_acc:.3f})')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/confusion_matrix.png', dpi=150)
plt.show()

def combine(h1, h2, key):
    return h1.history[key] + h2.history[key]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(combine(history_phase1, history_phase2, 'accuracy'),     label='Train')
ax1.plot(combine(history_phase1, history_phase2, 'val_accuracy'), label='Val')
ax1.axvline(x=len(history_phase1.history['accuracy']), color='gray',
            linestyle='--', label='Fine-tune start')
ax1.set_title('Accuracy — Phase 1 + Phase 2')
ax1.set_xlabel('Epoch')
ax1.legend()
ax1.grid(True)

ax2.plot(combine(history_phase1, history_phase2, 'loss'),     label='Train')
ax2.plot(combine(history_phase1, history_phase2, 'val_loss'), label='Val')
ax2.axvline(x=len(history_phase1.history['loss']), color='gray',
            linestyle='--', label='Fine-tune start')
ax2.set_title('Loss — Phase 1 + Phase 2')
ax2.set_xlabel('Epoch')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/training_history.png', dpi=150)
plt.show()

print(f"Plots saved to {RESULTS_DIR}/")

In [ ]:
# ── Cell 12: Real-time detection — MTCNN ───────────────
import cv2
import numpy as np
from mtcnn import MTCNN
from tensorflow import keras

# Load model
model = keras.models.load_model(MODEL_PATH)
detector = MTCNN()  # MTCNN face detector

CLASS_NAMES_DISPLAY = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']

EMOTION_COLORS = {
    'angry':    (0,   0,   255),
    'disgust':  (0,   128, 0),
    'fear':     (128, 0,   128),
    'happy':    (0,   255, 0),
    'neutral':  (255, 255, 0),
    'sad':      (255, 0,   0),
    'surprise': (0,   165, 255)
}

cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("Error: Could not open camera")
    exit()

frame_count  = 0
cached_faces = []   

print("Starting camera — press 'q' to quit")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    
    if frame_count % 5 == 0:
        detections = detector.detect_faces(rgb_frame)
        cached_faces = [
            d['box'] for d in detections
            if d['confidence'] > 0.90  
        ]

    for (x, y, w, h) in cached_faces:
        x, y = max(0, x), max(0, y)
        x2 = min(frame.shape[1], x + w)
        y2 = min(frame.shape[0], y + h)

    
        face_roi = rgb_frame[y:y2, x:x2]
        if face_roi.size == 0:
            continue
        face_roi = cv2.resize(face_roi, (IMG_SIZE, IMG_SIZE))
        face_roi = face_roi.astype('float32') / 255.0
        face_roi = np.expand_dims(face_roi, axis=0)  

        pred       = model.predict(face_roi, verbose=0)[0]
        idx        = np.argmax(pred)
        emotion    = CLASS_NAMES_DISPLAY[idx]
        confidence = pred[idx]
        color      = EMOTION_COLORS[emotion]

        cv2.rectangle(frame, (x, y), (x2, y2), color, 2)

 
        label = f"{emotion} {confidence*100:.1f}%"
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
        cv2.rectangle(frame, (x, y - th - 10), (x + tw + 6, y), color, -1)

       
        cv2.putText(frame, label, (x + 3, y - 6),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

        top3_idx = np.argsort(pred)[::-1][:3]
        for rank, i in enumerate(top3_idx):
            bar_label = f"{CLASS_NAMES_DISPLAY[i]}: {pred[i]*100:.0f}%"
            cv2.putText(frame, bar_label, (10, 25 + rank * 22),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1)

    cv2.imshow('Emotion Detector v2  (q to quit)', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
print("Emotion detection ended.")